# **1. Perkenalan Dataset**

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.

**Dataset yang digunakan: Heart Failure Prediction**

- **Sumber:** Kaggle — *fedesoriano/heart-failure-prediction*.
- **Jumlah data:** 918 baris, 11 fitur + 1 target.
- **Target:** `HeartDisease` (0 = tidak berisiko, 1 = berisiko penyakit jantung) — kasus **klasifikasi biner**.
- **Fitur numerik:** Age, RestingBP, Cholesterol, FastingBS, MaxHR, Oldpeak.
- **Fitur kategorikal:** Sex, ChestPainType, RestingECG, ExerciseAngina, ST_Slope.

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

In [ ]:
df = pd.read_csv("../heart_raw.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Jumlah missing value per kolom:")
df.isnull().sum()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

### 4.1 Distribusi kelas target

In [ ]:
sns.countplot(x="HeartDisease", data=df)
plt.title("Distribusi HeartDisease")
plt.show()

### 4.2 Distribusi fitur numerik

In [ ]:
df.select_dtypes(include="number").hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

### 4.3 Deteksi outlier (contoh: Cholesterol)

In [ ]:
sns.boxplot(x=df["Cholesterol"])
plt.title("Boxplot Cholesterol")
plt.show()

### 4.4 Matriks korelasi fitur numerik

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

**Insight EDA:**
- Distribusi kelas target relatif seimbang.
- Terdapat nilai 0 tidak wajar pada `Cholesterol` & `RestingBP` (indikasi missing terselubung/outlier).
- `Oldpeak`, `MaxHR`, dan `ST_Slope` cukup berkorelasi dengan target.
- Ada campuran fitur numerik & kategorikal → perlu scaling + encoding.

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

In [ ]:
# Pisahkan fitur & target
X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(include="object").columns.tolist()
print("Numerik:", numeric)
print("Kategorikal:", categorical)

In [ ]:
# 1) Tangani missing value (imputasi median) + 3) standarisasi fitur numerik
X[numeric] = SimpleImputer(strategy="median").fit_transform(X[numeric])
X[numeric] = StandardScaler().fit_transform(X[numeric])

In [ ]:
# 5) Encoding fitur kategorikal (one-hot)
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = enc.fit_transform(X[categorical])
X_enc = pd.concat(
    [X[numeric].reset_index(drop=True),
     pd.DataFrame(encoded, columns=enc.get_feature_names_out(categorical))],
    axis=1,
)
X_enc.head()

In [ ]:
# Split data latih & uji
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
# Simpan dataset hasil preprocessing (dipakai untuk pelatihan model)
X_enc.assign(HeartDisease=y).to_csv("heart_preprocessing.csv", index=False)
print("Tersimpan: heart_preprocessing.csv", X_enc.shape)

---
Seluruh tahapan preprocessing di atas telah dikonversi menjadi fungsi otomatis pada **`automate_handokobeni.py`**, yang dijalankan otomatis via **GitHub Actions** setiap ada perubahan.